# LeetCode 2029 — Stone Game IX bằng Python

Link đề: https://leetcode.com/problems/stone-game-ix/ · Độ khó: **Medium** · Chủ đề: Array, Math, Greedy, Counting, Game Theory

Skeleton LeetCode:

```python
class Solution:
    def stoneGameIX(self, stones: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Rút gọn trò chơi về **số dư mod 3**: chỉ cần biết `c0, c1, c2` = số viên có giá trị chia 3 dư `0, 1, 2`.
2. Viết **minimax có ghi nhớ** (trạng thái `(c0, c1, c2, tổng mod 3, lượt)`) để có đáp án chuẩn cho dữ liệu nhỏ.
3. Phân tích **chuỗi nước đi bị ép buộc** `1,1,2,1,2,...` (hoặc `2,2,1,2,1,...`) và vai trò của viên dư 0 như một nước **"đổi lượt"**.
4. Rút ra **công thức đóng** `O(1)`: nếu `c0` chẵn thì Alice thắng ⇔ `c1 ≥ 1` và `c2 ≥ 1`; nếu `c0` lẻ thì Alice thắng ⇔ `|c1 − c2| > 2`.
5. Kiểm chứng công thức bằng vét cạn trên **mọi** bộ `(c0, c1, c2)` nhỏ và trên mảng ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Có một hàng `n` viên đá, viên thứ `i` có giá trị `stones[i]`. Alice và Bob lần lượt chơi, **Alice đi trước**. Mỗi lượt, người chơi lấy **một viên bất kỳ** còn lại. Người lấy viên đá làm cho **tổng giá trị các viên đã lấy** (của cả hai người) **chia hết cho 3** thì **thua**. Nếu **hết đá** (mà chưa ai thua) thì **Bob thắng** — kể cả khi tới lượt Alice.

Hai người chơi tối ưu. Trả về `True` nếu Alice thắng.

**Ví dụ 1:** `[2,1]` → `True`. Alice lấy một viên, Bob buộc lấy viên còn lại, tổng `3` chia hết cho 3 ⇒ Bob thua.

**Ví dụ 2:** `[2]` → `False`. Alice lấy viên duy nhất (tổng 2), hết đá ⇒ Bob thắng.

**Ví dụ 3:** `[5,1,2,4,3]` → `False`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ stones[i] ≤ 10⁴`.

In [1]:
import random
import time
from functools import lru_cache
from itertools import product
from typing import List

examples = [([2, 1], True), ([2], False), ([5, 1, 2, 4, 3], False)]
def residues(stones):
    c = [0, 0, 0]
    for x in stones:
        c[x % 3] += 1
    return c
for st, exp in examples:
    print(st, "-> (c0, c1, c2) =", tuple(residues(st)), "| kỳ vọng", exp)

[2, 1] -> (c0, c1, c2) = (0, 1, 1) | kỳ vọng True
[2] -> (c0, c1, c2) = (0, 0, 1) | kỳ vọng False
[5, 1, 2, 4, 3] -> (c0, c1, c2) = (1, 2, 2) | kỳ vọng False


## 3. Kiến thức nền

### 3.1 Chỉ số dư mod 3 là quan trọng

`(a + b) mod 3 = ((a mod 3) + (b mod 3)) mod 3`. Điều kiện thua chỉ phụ thuộc tổng mod 3 ⇒ viên `5` và viên `2` "giống nhau" (cùng dư 2). Trạng thái trò chơi = số viên còn lại mỗi loại dư + tổng hiện tại mod 3 + ai đang đi.

In [2]:
print([(x, x % 3) for x in [5, 1, 2, 4, 3]])
print((5 + 1 + 2) % 3, ((5 % 3) + (1 % 3) + (2 % 3)) % 3)

[(5, 2), (1, 1), (2, 2), (4, 1), (3, 0)]
2 2


### 3.2 Minimax trên trò chơi hai người

Gọi `win(state)` = "người đang đi có thắng không?". Người đang đi thắng nếu **tồn tại** nước đi đưa về trạng thái mà **đối thủ thua**. Ở bài này kết quả khi hết đá phụ thuộc **danh tính** người chơi (Bob luôn thắng), nên ta lưu luôn "ai đang đi" và trả lời câu hỏi "Alice có thắng không?".

In [3]:
# Ví dụ minimax nhỏ: trò bốc 1 hoặc 2 que, ai bốc que cuối cùng thắng
@lru_cache(maxsize=None)
def nim12(k):
    return any(not nim12(k - t) for t in (1, 2) if t <= k)
print({k: nim12(k) for k in range(8)}, "-> người đi trước thua khi k chia hết cho 3")

{0: False, 1: True, 2: True, 3: False, 4: True, 5: True, 6: False, 7: True} -> người đi trước thua khi k chia hết cho 3


### 3.3 `functools.lru_cache` với trạng thái tuple

Hàm đệ quy có tham số là các số nguyên (hashable) có thể ghi nhớ bằng `@lru_cache(maxsize=None)`. Số trạng thái `(c0+1)(c1+1)(c2+1)·3·2` nhỏ nếu các `c` nhỏ — đủ để làm "đáp án chuẩn".

In [4]:
@lru_cache(maxsize=None)
def count_paths(a, b):
    return 1 if a == 0 or b == 0 else count_paths(a - 1, b) + count_paths(a, b - 1)
print(count_paths(10, 10), count_paths.cache_info().currsize, "trạng thái")

184756 120 trạng thái


### 3.4 Tính chẵn lẻ của số nước đi

Nếu một chuỗi `T` nước đi bị ép buộc, nước thứ `T + 1` thuộc về Alice khi `T` chẵn và thuộc về Bob khi `T` lẻ (Alice đi các nước lẻ: 1, 3, 5...).

In [5]:
for T in range(5):
    print(f"sau {T} nước, nước tiếp theo (thứ {T + 1}) là của", "Alice" if (T + 1) % 2 == 1 else "Bob")

sau 0 nước, nước tiếp theo (thứ 1) là của Alice
sau 1 nước, nước tiếp theo (thứ 2) là của Bob
sau 2 nước, nước tiếp theo (thứ 3) là của Alice
sau 3 nước, nước tiếp theo (thứ 4) là của Bob
sau 4 nước, nước tiếp theo (thứ 5) là của Alice


## 4. Ý tưởng nền tảng / trực giác

Theo dõi **tổng mod 3** `s`:
- Nếu `s = 1`: lấy viên dư 2 ⇒ `s = 0` ⇒ **thua**. Chỉ được lấy dư 1 (⇒ `s = 2`) hoặc dư 0 (⇒ `s = 1`).
- Nếu `s = 2`: lấy dư 1 ⇒ thua. Chỉ được lấy dư 2 (⇒ `s = 1`) hoặc dư 0.
- Lượt đầu `s = 0`: lấy dư 0 ⇒ tổng `0` chia hết ⇒ thua ngay; nên Alice phải mở bằng dư 1 hoặc dư 2.

Vậy nếu Alice mở bằng **1**, các viên khác 0 sau đó **bắt buộc** theo thứ tự `1, 2, 1, 2, ...` (tổng: `1 → 2 → 1 → 2 ...`). Viên dư 0 không đổi tổng — dùng nó như **"bỏ lượt"** (chuyển quyền đi cho đối thủ).

Tính tay ví dụ 1 `[2, 1]`: `c = (0, 1, 1)`. Alice mở 1 (`s = 1`), Bob chỉ còn viên dư 2 ⇒ buộc lấy ⇒ `s = 0` ⇒ Bob thua ✓.

Ví dụ 3 `[5,1,2,4,3]`: `c = (1, 2, 2)`. Mở 1: chuỗi `1,1,2,1,2` dùng hết 5 viên khác 0... có thêm một viên 0 làm đổi lượt. Kết quả (vét cạn ở dưới): Bob thắng.

In [6]:
def forced_sequence(c1, c2, first):
    """Chuỗi số dư khác 0 bị ép khi mở bằng `first` (không tính viên 0)."""
    seq, s = [], 0
    cnt = {1: c1, 2: c2}
    nxt = first
    while cnt[nxt] > 0:
        cnt[nxt] -= 1; seq.append(nxt); s = (s + nxt) % 3
        nxt = 1 if s == 1 else 2          # s = 1 -> phải lấy 1; s = 2 -> phải lấy 2
    return seq, cnt

print("c1=3, c2=2, mở 1:", forced_sequence(3, 2, 1))
print("c1=3, c2=2, mở 2:", forced_sequence(3, 2, 2))

c1=3, c2=2, mở 1: ([1, 1, 2, 1, 2], {1: 0, 2: 0})
c1=3, c2=2, mở 2: ([2, 2, 1], {1: 2, 2: 0})


## 5. Cách 1 — Brute Force / baseline: minimax trên tập viên đá (bitmask)

**Ý tưởng:** trạng thái là tập viên còn lại (bitmask) — tổng đã lấy và lượt suy ra được. `alice_wins(mask)`: nếu hết đá ⇒ `False`; người đang đi thử từng viên: nếu làm tổng chia hết cho 3 thì người đó thua ngay; ngược lại đệ quy.

In [7]:
def stone_brute(stones):
    n = len(stones); full = (1 << n) - 1; total = sum(stones)

    @lru_cache(maxsize=None)
    def alice_wins(mask):                       # mask = các viên CÒN LẠI
        if mask == 0:
            return False                        # hết đá: Bob thắng
        taken = n - bin(mask).count("1")
        alice_turn = taken % 2 == 0
        removed_sum = total - sum(stones[i] for i in range(n) if mask >> i & 1)
        outcomes = []
        for i in range(n):
            if mask >> i & 1:
                if (removed_sum + stones[i]) % 3 == 0:
                    outcomes.append(not alice_turn)      # người lấy thua
                else:
                    outcomes.append(alice_wins(mask ^ (1 << i)))
        return any(outcomes) if alice_turn else all(outcomes)
    return alice_wins(full)

for st, exp in examples:
    got = stone_brute(st); print(st, "->", got, "| kỳ vọng", exp); assert got == exp

[2, 1] -> True | kỳ vọng True
[2] -> False | kỳ vọng False
[5, 1, 2, 4, 3] -> False | kỳ vọng False


**Độ phức tạp:** `2ⁿ` trạng thái × `O(n)` ⇒ `O(2ⁿ·n)` (cộng thêm `O(n)` để tính tổng mỗi trạng thái). Chỉ dùng được với `n ≤ ~12`.

**Nút thắt:** hai viên cùng số dư là **hoán đổi được** — trạng thái chỉ cần `(c0, c1, c2)` chứ không cần biết viên nào.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — nén trạng thái:** `(c0, c1, c2, s, ai đi)` thay cho bitmask ⇒ số trạng thái đa thức.

**Insight 2 — chuỗi ép buộc:** sau khi Alice mở bằng `a ∈ {1, 2}`, các viên khác 0 bắt buộc theo chuỗi `a, a, b, a, b, ...` (với `b = 3 − a`). Gọi `T` = số viên khác 0 có thể lấy theo chuỗi này trước khi "kẹt" (viên cần lấy đã hết).

**Insight 3 — viên dư 0 là nước "đổi lượt":** cả hai đều có thể dùng viên 0 để chuyển quyền. Kết quả chỉ phụ thuộc **tính chẵn lẻ** của `c0`:
- `c0` **chẵn**: viên 0 "triệt tiêu" nhau (ai dùng, đối thủ dùng lại) ⇒ như không có. Mở bằng loại **ít hơn** (nhưng ≥ 1): Bob sẽ là người kẹt trước ⇒ Alice thắng ⇔ `c1 ≥ 1` **và** `c2 ≥ 1`.
- `c0` **lẻ**: đổi lượt một lần ⇒ Alice cần loại mình mở **nhiều hơn hẳn** loại kia (chênh `> 2`) để Bob vẫn bị kẹt mà **còn đá** ⇒ Alice thắng ⇔ `|c1 − c2| > 2`.

**Quy tắc "nếu… thì…"** (công thức đóng):
- **Nếu** `c0 % 2 == 0` **thì** `return c1 >= 1 and c2 >= 1`.
- **Ngược lại** `return abs(c1 - c2) > 2`.

Ta sẽ không tin công thức ngay mà kiểm bằng minimax ở Cách 2.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Minimax trên `(c0, c1, c2, s, lượt)` có ghi nhớ

**Giải thích từng đoạn:**
1. `f(c0, c1, c2, s, alice)` trả lời "Alice có thắng không".
2. Hết đá ⇒ `False`.
3. Với mỗi loại dư `r` còn viên: `ns = (s + r) % 3`; nếu `ns == 0` thì người đi thua ngay (kết quả = `not alice`); ngược lại đệ quy với lượt đổi.
4. Alice chọn `any`, Bob chọn `all` (Bob muốn Alice thua).

In [8]:
@lru_cache(maxsize=None)
def f_counts(c0, c1, c2, s, alice):
    if c0 + c1 + c2 == 0:
        return False
    res = []
    for r, c in ((0, c0), (1, c1), (2, c2)):
        if c == 0:
            continue
        ns = (s + r) % 3
        if ns == 0:
            res.append(not alice)               # người lấy viên này thua
        else:
            nc = [c0, c1, c2]; nc[r] -= 1
            res.append(f_counts(nc[0], nc[1], nc[2], ns, not alice))
    return any(res) if alice else all(res)

def stone_minimax_counts(stones):
    c0, c1, c2 = residues(stones)
    return f_counts(c0, c1, c2, 0, True)

for st, exp in examples:
    assert stone_minimax_counts(st) == exp
print([stone_minimax_counts(st) for st, _ in examples])

[True, False, False]


**Độ phức tạp Cách 2:** `O(c0·c1·c2)` trạng thái (×6), mỗi trạng thái `O(1)`. Đệ quy sâu tới `n` ⇒ với `n` lớn vượt giới hạn đệ quy ⇒ chỉ dùng làm chuẩn cho dữ liệu nhỏ.

### 7.2 Cách 3 — Mô phỏng chuỗi ép buộc cho hai cách mở

Với cách mở `a` (cần `c_a ≥ 1`):
1. Mô phỏng chuỗi khác 0 bị ép (`forced_sequence`) ⇒ `T` nước, còn lại `rest` viên khác 0.
2. Cộng `c0` nước "bỏ lượt" ⇒ tổng `M = T + c0` nước an toàn.
3. Nước thứ `M + 1` là của Bob ⇔ `M` lẻ. Alice thắng ⇔ `M` lẻ **và** vẫn còn viên (`rest > 0`), vì khi đó Bob buộc lấy một viên làm tổng chia hết cho 3. Nếu hết đá thì Bob thắng.

In [9]:
def stone_simulate(stones):
    c0, c1, c2 = residues(stones)
    def opening(first, c1, c2):
        if (c1 if first == 1 else c2) == 0:
            return False
        seq, left = forced_sequence(c1, c2, first)
        M = len(seq) + c0
        remaining = left[1] + left[2]
        return M % 2 == 1 and remaining > 0
    return opening(1, c1, c2) or opening(2, c1, c2)

for st, exp in examples:
    assert stone_simulate(st) == exp
print([stone_simulate(st) for st, _ in examples])

[True, False, False]


In [10]:
class SolutionSimulate:
    def stoneGameIX(self, stones: List[int]) -> bool:
        cnt = [0, 0, 0]
        for x in stones:
            cnt[x % 3] += 1
        c0 = cnt[0]
        def opening(a: int) -> bool:
            have = {1: cnt[1], 2: cnt[2]}
            if have[a] == 0:
                return False
            s, moves, nxt = 0, 0, a
            while have[nxt]:
                have[nxt] -= 1; moves += 1; s = (s + nxt) % 3
                nxt = 1 if s == 1 else 2
            return (moves + c0) % 2 == 1 and have[1] + have[2] > 0
        return opening(1) or opening(2)

print([SolutionSimulate().stoneGameIX(st) for st, _ in examples])

[True, False, False]


**Độ phức tạp Cách 3:** đếm `O(n)`, mô phỏng `O(n)` ⇒ `O(n)`, Space `O(1)`.

### 7.3 Cách 4 — Công thức đóng (bản nộp chính)

In [11]:
def stone_formula(stones):
    c0, c1, c2 = residues(stones)
    if c0 % 2 == 0:
        return c1 >= 1 and c2 >= 1
    return abs(c1 - c2) > 2

for st, exp in examples:
    assert stone_formula(st) == exp
print([stone_formula(st) for st, _ in examples])

[True, False, False]


**Độ phức tạp Cách 4:** `O(n)` để đếm, `O(1)` để quyết định. Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 3 `[5,1,2,4,3]`: `c = (c0, c1, c2) = (1, 2, 2)`.

| Cách mở | Chuỗi ép (khác 0) | T | M = T + c0 | còn viên khác 0? | Alice thắng? |
|---|---|---|---|---|---|
| 1 | 1, 1, 2, 1? (hết 1) → `1,1,2` | 3 | 4 | còn 1 viên dư 2 | M chẵn ⇒ **không** |
| 2 | 2, 2, 1, 2? (hết 2) → `2,2,1` | 3 | 4 | còn 1 viên dư 1 | M chẵn ⇒ **không** |

⇒ `False` ✓. Công thức: `c0 = 1` lẻ, `|2 − 2| = 0 ≤ 2` ⇒ `False` ✓.

In [12]:
def trace_simulate(stones):
    c0, c1, c2 = residues(stones)
    print(f"(c0, c1, c2) = ({c0}, {c1}, {c2})")
    result = False
    for first in (1, 2):
        if (c1 if first == 1 else c2) == 0:
            print(f"mở {first}: không có viên dư {first}"); continue
        seq, left = forced_sequence(c1, c2, first)
        M = len(seq) + c0; rem = left[1] + left[2]
        win = M % 2 == 1 and rem > 0
        print(f"mở {first}: chuỗi ép {seq} (T={len(seq)}), M = T + c0 = {M}, còn {rem} viên khác 0 "
              f"-> nước {M + 1} của {'Bob' if M % 2 else 'Alice'} -> Alice {'THẮNG' if win else 'không thắng'}")
        result |= win
    return result

print("Đáp án:", trace_simulate([5, 1, 2, 4, 3]))
print()
print("Đáp án:", trace_simulate([2, 1]))
print()
print("Đáp án:", trace_simulate([3, 1, 1, 1, 1, 2]))

(c0, c1, c2) = (1, 2, 2)
mở 1: chuỗi ép [1, 1, 2] (T=3), M = T + c0 = 4, còn 1 viên khác 0 -> nước 5 của Alice -> Alice không thắng
mở 2: chuỗi ép [2, 2, 1] (T=3), M = T + c0 = 4, còn 1 viên khác 0 -> nước 5 của Alice -> Alice không thắng
Đáp án: False

(c0, c1, c2) = (0, 1, 1)
mở 1: chuỗi ép [1] (T=1), M = T + c0 = 1, còn 1 viên khác 0 -> nước 2 của Bob -> Alice THẮNG
mở 2: chuỗi ép [2] (T=1), M = T + c0 = 1, còn 1 viên khác 0 -> nước 2 của Bob -> Alice THẮNG
Đáp án: True

(c0, c1, c2) = (1, 4, 1)
mở 1: chuỗi ép [1, 1, 2, 1] (T=4), M = T + c0 = 5, còn 1 viên khác 0 -> nước 6 của Bob -> Alice THẮNG
mở 2: chuỗi ép [2] (T=1), M = T + c0 = 2, còn 4 viên khác 0 -> nước 3 của Alice -> Alice không thắng
Đáp án: True


Để thấy **cây trò chơi** thật sự (Cách 2), hàm `trace_tree` in mọi nhánh tới độ sâu nhỏ cho một trạng thái nhỏ. Mỗi dòng: ai đi, lấy viên dư mấy, tổng mới, và kết quả "Alice thắng?" của nhánh đó. Hãy để ý Bob (lượt chẵn) luôn chọn nhánh làm Alice thua nếu có.

In [13]:
def trace_tree(c0, c1, c2, s=0, alice=True, depth=0, max_depth=4):
    who = "Alice" if alice else "Bob"
    if c0 + c1 + c2 == 0:
        print("  " * depth + "hết đá -> Bob thắng"); return False
    results = []
    for r, c in ((0, c0), (1, c1), (2, c2)):
        if c == 0: continue
        ns = (s + r) % 3
        if ns == 0:
            res = not alice
            print("  " * depth + f"{who} lấy dư {r}: tổng mod 3 = 0 -> {who} thua (Alice thắng? {res})")
        else:
            nc = [c0, c1, c2]; nc[r] -= 1
            res = f_counts(nc[0], nc[1], nc[2], ns, not alice)
            print("  " * depth + f"{who} lấy dư {r}: tổng mod 3 = {ns}, còn {tuple(nc)} (Alice thắng? {res})")
            if depth + 1 < max_depth:
                trace_tree(nc[0], nc[1], nc[2], ns, not alice, depth + 1, max_depth)
        results.append(res)
    return any(results) if alice else all(results)

print("Cây trò chơi của [2, 1] (c = (0, 1, 1)):")
print("=> Alice thắng?", trace_tree(0, 1, 1))
print()
print("Cây trò chơi của c = (1, 1, 1), ví dụ [3, 1, 2] (2 tầng đầu):")
print("=> Alice thắng?", trace_tree(1, 1, 1, max_depth=2))

Cây trò chơi của [2, 1] (c = (0, 1, 1)):
Alice lấy dư 1: tổng mod 3 = 1, còn (0, 0, 1) (Alice thắng? True)
  Bob lấy dư 2: tổng mod 3 = 0 -> Bob thua (Alice thắng? True)
Alice lấy dư 2: tổng mod 3 = 2, còn (0, 1, 0) (Alice thắng? True)
  Bob lấy dư 1: tổng mod 3 = 0 -> Bob thua (Alice thắng? True)
=> Alice thắng? True

Cây trò chơi của c = (1, 1, 1), ví dụ [3, 1, 2] (2 tầng đầu):
Alice lấy dư 0: tổng mod 3 = 0 -> Alice thua (Alice thắng? False)
Alice lấy dư 1: tổng mod 3 = 1, còn (1, 0, 1) (Alice thắng? False)
  Bob lấy dư 0: tổng mod 3 = 1, còn (0, 0, 1) (Alice thắng? False)
  Bob lấy dư 2: tổng mod 3 = 0 -> Bob thua (Alice thắng? True)
Alice lấy dư 2: tổng mod 3 = 2, còn (1, 1, 0) (Alice thắng? False)
  Bob lấy dư 0: tổng mod 3 = 2, còn (0, 1, 0) (Alice thắng? False)
  Bob lấy dư 1: tổng mod 3 = 0 -> Bob thua (Alice thắng? True)
=> Alice thắng? False


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def stoneGameIX(self, stones: List[int]) -> bool:
        c = [0, 0, 0]
        for x in stones:
            c[x % 3] += 1                 # chỉ số dư mod 3 quan trọng
        if c[0] % 2 == 0:                 # viên dư 0 "triệt tiêu" nhau
            return c[1] >= 1 and c[2] >= 1
        return abs(c[1] - c[2]) > 2       # một lần đổi lượt: cần chênh lệch > 2

for st, exp in examples:
    assert Solution().stoneGameIX(st) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Ta chứng minh ngắn (và kiểm chứng đầy đủ bằng máy ở mục 12).

**Nước đi hợp lệ "không thua ngay".** Khi tổng `s = 1`, chỉ có thể lấy dư 0 hoặc dư 1; khi `s = 2`, chỉ dư 0 hoặc dư 2. Nếu hết các viên an toàn, người đang đi buộc lấy viên làm tổng chia hết cho 3 (thua), **trừ khi** đã hết đá (Bob thắng).

**Trường hợp `c0` chẵn.** Lập luận "bắt chước": mỗi khi đối thủ lấy viên 0, ta cũng lấy viên 0 (tổng không đổi, số viên 0 còn chẵn) — nên có thể coi như `c0 = 0`.
- Nếu `c1 ≥ 1` và `c2 ≥ 1`: Alice mở bằng loại **ít hơn** (giả sử `c1 ≤ c2`, mở 1). Chuỗi ép `1, 1, 2, 1, 2, ...`: Bob luôn cần dư 1, Alice cần dư 2. Vì dư 1 cạn trước (hoặc cùng lúc) ⇒ Bob là người kẹt khi còn ít nhất một viên dư 2 ⇒ Bob thua. (Nếu `c1 = c2` thì sau `1` của Alice và `1` của Bob, ... Bob vẫn kẹt trước vì Alice đã dùng một viên 1 ở nước mở.)
- Nếu một trong hai bằng 0: chuỗi ép chỉ có một loại (`a, a`, rồi cần `b` không có) — người kẹt là Alice ở nước 3, hoặc đá hết ⇒ Bob thắng.

**Trường hợp `c0` lẻ.** Viên 0 cuối cùng là một lần **đổi lượt** thực sự ⇒ người kẹt bị đảo. Alice chỉ thắng nếu mở bằng loại nhiều hơn và chênh lệch đủ lớn để sau chuỗi `a, a, b, a, b, ..., a` (tiêu thụ `k + 2` viên `a` và `k` viên `b`) **vẫn còn** viên `a` khiến Bob kẹt mà không hết đá: điều này cần `c_a − c_b > 2`.

Bảng kiểm chứng dưới đây so công thức với minimax (Cách 2) trên **mọi** `0 ≤ c0, c1, c2 ≤ 12`.

In [15]:
mismatch = 0
for c0 in range(13):
    for c1 in range(13):
        for c2 in range(13):
            if c0 + c1 + c2 == 0: continue
            mm = f_counts(c0, c1, c2, 0, True)
            fm = (c1 >= 1 and c2 >= 1) if c0 % 2 == 0 else abs(c1 - c2) > 2
            mismatch += mm != fm
print("Số bộ (c0, c1, c2) <= 12 mà công thức khác minimax:", mismatch)

Số bộ (c0, c1, c2) <= 12 mà công thức khác minimax: 0


## 11. Phân tích Time / Space

Gọi `n = len(stones)`.

| Cách | Time | Space |
|---|---|---|
| 1. Minimax bitmask | `O(2ⁿ · n)` | `O(2ⁿ)` |
| 2. Minimax theo số dư | `O(c0·c1·c2)` | `O(c0·c1·c2)` + đệ quy sâu `n` |
| 3. Mô phỏng chuỗi ép | `O(n)` | `O(1)` |
| 4. Công thức | `O(n)` (đếm) + `O(1)` | `O(1)` |

In [16]:
rng = random.Random(2029)
big = [rng.randint(1, 10**4) for _ in range(10**5)]
for name, fn in [("simulate", stone_simulate), ("formula", stone_formula),
                 ("SolutionSimulate", lambda s: SolutionSimulate().stoneGameIX(s)), ("Solution", lambda s: Solution().stoneGameIX(s))]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:16s}: {r} ({time.perf_counter() - t:.3f}s), c = {residues(big)}")

simulate        : True (0.014s), c = [33498, 33365, 33137]
formula         : True (0.003s), c = [33498, 33365, 33137]
SolutionSimulate: True (0.014s), c = [33498, 33365, 33137]
Solution        : True (0.003s), c = [33498, 33365, 33137]


## 12. Bộ test

- Ví dụ đề.
- Đối chiếu minimax bitmask (luật gốc, không rút gọn) với mọi cách trên **mọi** mảng giá trị trong `{1,2,3}` độ dài `1..7` (mỗi giá trị đại diện một số dư).
- Đối chiếu minimax theo số dư với công thức và mô phỏng trên mọi `(c0, c1, c2)` nhỏ (đã làm ở mục 10) và trên mảng ngẫu nhiên.

In [17]:
fns = {
    "brute": stone_brute, "minimax_counts": stone_minimax_counts, "simulate": stone_simulate,
    "formula": stone_formula, "Solution": lambda s: Solution().stoneGameIX(s),
    "SolutionSimulate": lambda s: SolutionSimulate().stoneGameIX(s),
}
for st, exp in examples:
    for name, fn in fns.items():
        assert fn(st) == exp, (name, st)
cnt = 0
for n in range(1, 8):
    for tup in product((1, 2, 3), repeat=n):
        want = stone_brute(list(tup))
        for name, fn in fns.items():
            assert fn(list(tup)) == want, (name, tup)
        cnt += 1
print(f"Ví dụ + vét cạn {cnt} mảng (giá trị 1..3, n <= 7): OK")

Ví dụ + vét cạn 3279 mảng (giá trị 1..3, n <= 7): OK


In [18]:
rng = random.Random(42)
for _ in range(300):
    st = [rng.randint(1, 10**4) for _ in range(rng.randint(1, 40))]
    want = stone_minimax_counts(st)
    assert stone_formula(st) == stone_simulate(st) == Solution().stoneGameIX(st) == want
for _ in range(100):
    st = [rng.randint(1, 20) for _ in range(rng.randint(1, 10))]
    assert stone_brute(st) == Solution().stoneGameIX(st)
print("400 test ngẫu nhiên (giá trị lớn): OK")

400 test ngẫu nhiên (giá trị lớn): OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua viên dư 0

Coi viên dư 0 là vô nghĩa (chỉ dùng điều kiện `c1 ≥ 1 and c2 ≥ 1`) sẽ sai khi `c0` lẻ.

In [19]:
def wrong_ignore_zero(stones):
    c0, c1, c2 = residues(stones)
    return c1 >= 1 and c2 >= 1                    # SAI: quên tính chẵn lẻ của c0

st = [3, 1, 2]
print("sai:", wrong_ignore_zero(st), "| đúng:", stone_brute(st))

sai: True | đúng: False


In [20]:
print("sửa:", Solution().stoneGameIX(st))

sửa: False


### Lỗi 2 — Chỉ thử mở bằng viên dư 1

Có trường hợp Alice phải mở bằng dư 2 (ví dụ không có viên dư 1 nào).

In [21]:
def wrong_only_open_1(stones):
    c0, c1, c2 = residues(stones)
    if c1 == 0: return False                       # SAI: không thử mở bằng 2
    seq, left = forced_sequence(c1, c2, 1)
    M = len(seq) + c0
    return M % 2 == 1 and left[1] + left[2] > 0

st = [3, 2, 2, 2, 2]
print("sai:", wrong_only_open_1(st), "| đúng:", stone_brute(st))

sai: False | đúng: True


In [22]:
print("sửa (thử cả hai cách mở):", stone_simulate(st))

sửa (thử cả hai cách mở): True


### Lỗi 3 — Quên luật "hết đá thì Bob thắng"

Nếu coi hết đá là "người vừa đi thắng" hoặc bỏ qua luật này, ví dụ 2 `[2]` sẽ thành Alice thắng.

In [23]:
def wrong_exhaust_rule(stones):
    n = len(stones); total = sum(stones)
    @lru_cache(maxsize=None)
    def alice_wins(mask):
        if mask == 0:
            taken = n
            return taken % 2 == 1                  # SAI: coi người đi cuối (Alice nếu n lẻ) thắng
        taken = n - bin(mask).count("1"); alice_turn = taken % 2 == 0
        removed = total - sum(stones[i] for i in range(n) if mask >> i & 1)
        outs = [(not alice_turn) if (removed + stones[i]) % 3 == 0 else alice_wins(mask ^ (1 << i))
                for i in range(n) if mask >> i & 1]
        return any(outs) if alice_turn else all(outs)
    return alice_wins((1 << n) - 1)

print("sai:", wrong_exhaust_rule([2]), "| đúng:", stone_brute([2]))

sai: True | đúng: False


In [24]:
print("sửa (hết đá -> Bob thắng):", stone_brute([2]), Solution().stoneGameIX([2]))

sửa (hết đá -> Bob thắng): False False


### Lỗi 4 — Dùng `>= 2` thay vì `> 2` khi `c0` lẻ

Chênh lệch đúng bằng 2 vẫn là Bob thắng: Alice mở bằng loại nhiều hơn, chuỗi ép dùng hết loại ít hơn và đúng hai viên dư của loại nhiều hơn ⇒ **hết đá**.

In [25]:
def wrong_ge2(stones):
    c0, c1, c2 = residues(stones)
    return (c1 >= 1 and c2 >= 1) if c0 % 2 == 0 else abs(c1 - c2) >= 2   # SAI: >= 2

st = [3, 1, 1]
print("sai:", wrong_ge2(st), "| đúng:", stone_brute(st))

sai: True | đúng: False


In [26]:
print("sửa (> 2):", Solution().stoneGameIX(st))

sửa (> 2): False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Trò chơi có điều kiện "chia hết cho k"** ⇒ rút gọn về số dư mod `k`, trạng thái = số lượng mỗi loại dư.
- **Phần tử "trung tính"** (không đổi trạng thái, như dư 0) ⇒ đóng vai **đổi lượt**, chỉ tính **chẵn lẻ** của số lượng.
- Khi lựa chọn bị **ép buộc** sau nước mở ⇒ mô phỏng chuỗi ép, đếm nước, xét chẵn lẻ.
- Luôn viết **minimax có ghi nhớ** cho dữ liệu nhỏ để kiểm công thức — game theory rất dễ suy luận sai.
- Đọc kỹ luật kết thúc khi **hết tài nguyên** (ở đây Bob thắng).

## 15. Biến thể của bài

1. **Stone Game I–VIII (LeetCode 877, 1140, 1406, 1510, 1563, 1686, 1690, 1872):** đa số là DP minimax trên đoạn / tiền tố.
2. **Đổi mod 3 thành mod `k`** khác: phân tích chuỗi ép phức tạp hơn; minimax theo số dư (Cách 2) vẫn dùng được nguyên khung, chỉ đổi số loại dư.
3. **Đổi luật hết đá** thành "người đi cuối thắng" — công thức thay đổi.
4. **Trả về nước mở tối ưu** cho Alice (bài tập 5).

Minh họa biến thể 3 (tự dựng): hết đá thì **Alice** thắng. Ta chỉ cần sửa trạng thái cuối của minimax rồi quan sát.

In [27]:
@lru_cache(maxsize=None)
def f_alice_on_exhaust(c0, c1, c2, s, alice):
    if c0 + c1 + c2 == 0:
        return True                               # luật mới: hết đá -> Alice thắng
    res = []
    for r, c in ((0, c0), (1, c1), (2, c2)):
        if c == 0: continue
        ns = (s + r) % 3
        if ns == 0: res.append(not alice)
        else:
            nc = [c0, c1, c2]; nc[r] -= 1
            res.append(f_alice_on_exhaust(nc[0], nc[1], nc[2], ns, not alice))
    return any(res) if alice else all(res)

table = {(c0, c1, c2): f_alice_on_exhaust(c0, c1, c2, 0, True) for c0 in range(3) for c1 in range(4) for c2 in range(4) if c0 + c1 + c2}
print("Luật mới, một số kết quả:", {k: v for k, v in list(table.items())[:10]})
print("[2] theo luật mới ->", f_alice_on_exhaust(0, 0, 1, 0, True), "(luật gốc: False)")

Luật mới, một số kết quả: {(0, 0, 1): True, (0, 0, 2): True, (0, 0, 3): False, (0, 1, 0): True, (0, 1, 1): True, (0, 1, 2): True, (0, 1, 3): True, (0, 2, 0): True, (0, 2, 1): True, (0, 2, 2): True}
[2] theo luật mới -> True (luật gốc: False)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax bitmask | luật gốc, không rút gọn | `O(2ⁿ·n)` | `O(2ⁿ)` | chuẩn tuyệt đối, `n ≤ 12` |
| 2. Minimax số dư | trạng thái `(c0,c1,c2,s,lượt)` | `O(c0·c1·c2)` | như trên | kiểm công thức |
| 3. Mô phỏng chuỗi ép | thử hai cách mở | `O(n)` | `O(1)` | hiểu được cơ chế |
| 4. Công thức | chẵn lẻ `c0`, chênh lệch | `O(n)` | `O(1)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `stoneGameIX([1,1,7,10,8,17,10,20,2,10])`.
2. **Tính tay:** `[3, 1, 1, 1, 1, 2]` — tính `c`, chạy hai cách mở như mục 8.
3. **Điền TODO:** `residue_counts(stones)` trả về `(c0, c1, c2)`.
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `best_opening(stones)` trả về `1` hoặc `2` (số dư của viên Alice nên mở để thắng; nếu cả hai đều thắng trả số nhỏ hơn) hoặc `None` nếu Alice thua. Kiểm bằng minimax theo số dư.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def residue_counts(stones):
    # TODO
    pass

In [29]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_stone_game_ix(stones):
    # TODO
    pass

In [30]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def best_opening(stones):
    # TODO: với a in (1, 2): còn viên dư a? sau khi lấy, Bob đi ở trạng thái (c0, ..., s=a) — Alice thắng?
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Số dư: `1,1,1,1,2,2,1,2,2,1`. (2) `c = (0, 6, 4)`. (3) `c0` chẵn, cả hai loại ≥ 1 ⇒ `True`.

**Bài 2.** (1) `c = (1, 4, 1)`. (2) `c0` lẻ, `|4 − 1| = 3 > 2`. (3) Mở bằng 1: chuỗi `1,1,2,1` (hết 2) T = 4, M = 5 lẻ, còn 1 viên dư 1 ⇒ Alice thắng.

**Bài 3.** (1) `c = [0, 0, 0]`. (2) `c[x % 3] += 1`. (3) Trả tuple.

**Bài 4.** (1) Đếm. (2) `c0` chẵn ⇒ `c1 ≥ 1 and c2 ≥ 1`. (3) Ngược lại `abs(c1 − c2) > 2`.

**Bài 5.** (1) Dùng `f_counts(c0, c1 − 1, c2, 1, False)` cho cách mở 1. (2) Tương tự `f_counts(c0, c1, c2 − 1, 2, False)` cho cách mở 2. (3) Trả cách mở đầu tiên cho `True`.

## 19. Đáp án tham khảo

In [31]:
# Bài 1, 2
assert Solution().stoneGameIX([1, 1, 7, 10, 8, 17, 10, 20, 2, 10]) is True
assert Solution().stoneGameIX([3, 1, 1, 1, 1, 2]) is True
print("Bài 1: True | Bài 2: True (xem trace ở mục 8)")

Bài 1: True | Bài 2: True (xem trace ở mục 8)


In [32]:
# Bài 3, 4
def residue_counts_ans(stones):
    c = [0, 0, 0]
    for x in stones: c[x % 3] += 1
    return tuple(c)

def my_stone_game_ix_ans(stones):
    c0, c1, c2 = residue_counts_ans(stones)
    return (c1 >= 1 and c2 >= 1) if c0 % 2 == 0 else abs(c1 - c2) > 2

rng = random.Random(3)
for _ in range(300):
    st = [rng.randint(1, 30) for _ in range(rng.randint(1, 25))]
    assert my_stone_game_ix_ans(st) == stone_minimax_counts(st)
print("Bài 3, 4: OK;", residue_counts_ans([5, 1, 2, 4, 3]))

Bài 3, 4: OK; (1, 2, 2)


In [33]:
# Bài 5
def best_opening_ans(stones):
    c0, c1, c2 = residue_counts_ans(stones)
    if c1 and f_counts(c0, c1 - 1, c2, 1, False):
        return 1
    if c2 and f_counts(c0, c1, c2 - 1, 2, False):
        return 2
    return None

def best_opening_brute(stones):
    # thử thật: Alice lấy một viên cụ thể có dư a, rồi minimax bitmask phần còn lại
    n = len(stones)
    for a in (1, 2):
        for i in range(n):
            if stones[i] % 3 == a:
                rest = stones[:i] + stones[i + 1:]
                # sau nước mở: tổng = a, tới lượt Bob; Alice thắng nếu Bob (người đi trước của phần còn lại) thua
                @lru_cache(maxsize=None)
                def alice_wins(mask, m=len(rest)):
                    if mask == 0: return False
                    taken = m - bin(mask).count("1"); alice_turn = taken % 2 == 1
                    removed = a + sum(rest) - sum(rest[j] for j in range(m) if mask >> j & 1)
                    outs = [(not alice_turn) if (removed + rest[j]) % 3 == 0 else alice_wins(mask ^ (1 << j))
                            for j in range(m) if mask >> j & 1]
                    return any(outs) if alice_turn else all(outs)
                if rest and alice_wins((1 << len(rest)) - 1):
                    return a
                break                                  # các viên cùng dư a cho cùng kết quả
    return None

rng = random.Random(5)
for _ in range(150):
    st = [rng.randint(1, 9) for _ in range(rng.randint(1, 8))]
    assert best_opening_ans(st) == best_opening_brute(st)
    assert (best_opening_ans(st) is not None) == Solution().stoneGameIX(st)
print("Bài 5: OK; [3,2,2,2,2] ->", best_opening_ans([3, 2, 2, 2, 2]), "| [2,1] ->", best_opening_ans([2, 1]))

Bài 5: OK; [3,2,2,2,2] -> 2 | [2,1] -> 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
c = [0, 0, 0]
for x in stones: c[x % 3] += 1
if c[0] % 2 == 0:
    return c[1] >= 1 and c[2] >= 1
return abs(c[1] - c[2]) > 2
```

**Template kiểm chứng (minimax theo số dư):** `f(c0, c1, c2, s, alice)`; hết đá ⇒ `False`; nước làm `s' = 0` ⇒ người đi thua; Alice `any`, Bob `all`.

In [34]:
def template(stones):
    c = [0, 0, 0]
    for x in stones: c[x % 3] += 1
    return (c[1] >= 1 and c[2] >= 1) if c[0] % 2 == 0 else abs(c[1] - c[2]) > 2
print([template(st) for st, _ in examples])

[True, False, False]


**Checklist:**

- [ ] Chỉ dùng số dư mod 3 (`c0, c1, c2`).
- [ ] Tính chẵn lẻ của `c0` (viên dư 0 là nước đổi lượt).
- [ ] `c0` chẵn: cần **cả** `c1 ≥ 1` và `c2 ≥ 1`.
- [ ] `c0` lẻ: `|c1 − c2| > 2` (nghiêm ngặt).
- [ ] Nhớ luật: hết đá thì Bob thắng.
- [ ] Kiểm công thức bằng minimax trên dữ liệu nhỏ trước khi tin.